# Cloudburst Prediction REST API
## CNN-GAF Real-Time Forecasting Service — Uttarakhand, India

**Paper:** Precision forecasting of cloudbursts with CNNs and GAF for real-time disaster response
**DOI:** [10.1007/s40808-026-02826-4](https://doi.org/10.1007/s40808-026-02826-4)

This notebook implements a production-ready REST API using **FastAPI** that:
- Fetches live meteorological data from the IMD AWS endpoint
- Applies GAF encoding to the 6-day time-series
- Returns 6-hour and 12-hour cloudburst probabilities with risk levels

### Endpoints
| Method | Path | Description |
|--------|------|-------------|
| `GET` | `/health` | Service health check |
| `POST` | `/predict` | Cloudburst prediction for a given station |
| `GET` | `/stations` | Sample IMD station reference |

## Install Dependencies

In [ ]:
# Run once to install required packages
# !pip install fastapi uvicorn[standard] pydantic pyts tensorflow requests pandas lxml nest-asyncio

## Imports

In [ ]:
import os
import datetime
import logging
import warnings
import numpy as np
import pandas as pd
import requests
import nest_asyncio
import uvicorn
from threading import Thread

from fastapi import FastAPI, HTTPException, status
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field

from pyts.image import GramianAngularField
from keras.models import load_model

warnings.filterwarnings('ignore')
nest_asyncio.apply()  # Allows uvicorn to run inside a Jupyter kernel

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s — %(message)s'
)
logger = logging.getLogger('cloudburst_api')

## Configuration

In [ ]:
# ── Model paths
MODEL_6H_PATH  = 'Model6.h5'
MODEL_12H_PATH = 'Model12.h5'

# ── IMD AWS endpoint
IMD_BASE_URL   = 'http://aws.imd.gov.in:8091/AWS/dataview.php'
STATE          = 'UTTARAKHAND'
LOOKBACK_DAYS  = 6
REQUEST_TIMEOUT = 30  # seconds

# ── Feature names (must match IMD table columns exactly)
FEATURES = [
    "RAIN FALL CUM. SINCE 0300 UTC (mm)",
    "TEMP. (\'C)",
    "RH (%)",
    "WIND SPEED 10 m (Kt)",
    "SLP (hPa)",
    "MSLP (hPa / gpm)"
]

# ── API server settings
API_HOST = '0.0.0.0'
API_PORT = 8000

## Load Models

In [ ]:
logger.info("Loading pre-trained models...")

try:
    model_6h  = load_model(MODEL_6H_PATH)
    model_12h = load_model(MODEL_12H_PATH)
    logger.info("Models loaded successfully.")
    print(f"6H  model input shape: {model_6h.input_shape}")
    print(f"12H model input shape: {model_12h.input_shape}")
except Exception as e:
    logger.error(f"Failed to load models: {e}")
    raise

## Data Fetching

In [ ]:
def fetch_imd_data(district: str, station: str) -> pd.DataFrame:
    """
    Fetch meteorological time-series from IMD AWS endpoint.

    Args:
        district: IMD district name (e.g. 'DEHRADUN')
        station:  IMD AWS station code (e.g. 'MUSSOORIE(UKG)_UKG')

    Returns:
        DataFrame with columns matching FEATURES list.

    Raises:
        HTTPException 502: IMD API unreachable or an error.
        HTTPException 422: No usable data tables in the response.
    """
    today     = datetime.date.today()
    from_date = today - datetime.timedelta(days=LOOKBACK_DAYS)

    params = {
        'a': 'AWS', 'b': STATE,
        'c': district, 'd': station,
        'e': str(from_date), 'f': str(today),
        'g': 'ALL_HOUR', 'h': 'ALL_MINUTE'
    }

    logger.info(f"Fetching IMD data: district={district}, station={station}, "
                f"range={from_date} → {today}")

    try:
        resp = requests.get(IMD_BASE_URL, params=params, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
    except requests.exceptions.Timeout:
        raise HTTPException(
            status_code=status.HTTP_504_GATEWAY_TIMEOUT,
            detail="IMD API request timed out. Try again shortly."
        )
    except requests.exceptions.ConnectionError:
        raise HTTPException(
            status_code=status.HTTP_502_BAD_GATEWAY,
            detail="Cannot reach IMD AWS endpoint. Check network or endpoint availability."
        )
    except requests.exceptions.HTTPError as e:
        raise HTTPException(
            status_code=status.HTTP_502_BAD_GATEWAY,
            detail=f"IMD API returned an error: {e}"
        )

    try:
        tables = pd.read_html(resp.text)
    except ValueError:
        raise HTTPException(
            status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
            detail="IMD response contained no parseable data tables."
        )

    if not tables:
        raise HTTPException(
            status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
            detail="No data returned for the specified district/station."
        )

    df = tables[0]
    logger.info(f"IMD data fetched: {df.shape[0]} rows, {df.shape[1]} columns")
    return df

## GAF Transformation

In [ ]:
def generate_gaf(df: pd.DataFrame) -> np.ndarray:
    """
    Convert a meteorological DataFrame into a stack of GAF images.

    Each feature column is independently encoded into a 256×256 GAF image
    using the Gramian Angular Summation Field method. NaN values are imputed
    with the column mean before encoding.

    Args:
        df: DataFrame with at least the columns listed in FEATURES.

    Returns:
        np.ndarray of shape (1, 6, 256, 256) — ready for model.predict().

    Raises:
        HTTPException 422: Missing feature column or insufficient data.
    """
    gaf_encoder = GramianAngularField(image_size=256, method='summation')
    gaf_images  = []

    for feature in FEATURES:
        if feature not in df.columns:
            raise HTTPException(
                status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
                detail=f"Feature column '{feature}' not found in IMD data. "
                       f"Available columns: {list(df.columns)}"
            )

        series = df[feature].copy()
        n_valid = series.notna().sum()

        if n_valid < 2:
            raise HTTPException(
                status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
                detail=f"Feature '{feature}' has fewer than 2 valid readings. "
                       f"Cannot perform GAF encoding."
            )

        series = series.fillna(series.mean()).values
        image  = gaf_encoder.transform([series])
        gaf_images.append(image[0])

    return np.array([gaf_images])  # shape: (1, 6, 256, 256)

## Request & Response Schemas

In [ ]:
class PredictionRequest(BaseModel):
    district: str = Field(
        ...,
        example="DEHRADUN",
        description="IMD district name (uppercase)"
    )
    station: str = Field(
        ...,
        example="MUSSOORIE(UKG)_UKG",
        description="IMD AWS station identifier"
    )

class RiskLevel(str):
    HIGH     = "HIGH"
    MODERATE = "MODERATE"
    LOW      = "LOW"

class PredictionResponse(BaseModel):
    district:          str
    station:           str
    query_date:        str
    data_from:         str
    prediction_6h:     float = Field(..., description="Cloudburst probability (%) — 6-hour lead")
    prediction_12h:    float = Field(..., description="Cloudburst probability (%) — 12-hour lead")
    risk_level_6h:     str   = Field(..., description="HIGH / MODERATE / LOW")
    risk_level_12h:    str   = Field(..., description="HIGH / MODERATE / LOW")
    model_version:     str   = "1.0.0"

class HealthResponse(BaseModel):
    status:        str
    models_loaded: bool
    timestamp:     str

## Prediction Logic

In [ ]:
def risk_label(probability_pct: float) -> str:
    """Map probability percentage to a categorical risk level."""
    if probability_pct >= 75.0:
        return "HIGH"
    elif probability_pct >= 40.0:
        return "MODERATE"
    else:
        return "LOW"


def run_prediction(district: str, station: str) -> PredictionResponse:
    """
    Full inference pipeline:
    1. Fetch IMD data
    2. GAF encode
    3. Run both models
    4. Return structured response
    """
    today     = datetime.date.today()
    from_date = today - datetime.timedelta(days=LOOKBACK_DAYS)

    df       = fetch_imd_data(district, station)
    gaf_data = generate_gaf(df)

    prob_6h  = float(model_6h.predict(gaf_data,  verbose=0)[0][0])
    prob_12h = float(model_12h.predict(gaf_data, verbose=0)[0][0])

    pct_6h  = round(prob_6h  * 100, 2)
    pct_12h = round(prob_12h * 100, 2)

    logger.info(f"Prediction — {district}/{station}: "
                f"6H={pct_6h}% ({risk_label(pct_6h)})  "
                f"12H={pct_12h}% ({risk_label(pct_12h)})")

    return PredictionResponse(
        district       = district,
        station        = station,
        query_date     = str(today),
        data_from      = str(from_date),
        prediction_6h  = pct_6h,
        prediction_12h = pct_12h,
        risk_level_6h  = risk_label(pct_6h),
        risk_level_12h = risk_label(pct_12h),
    )

## API Application

In [ ]:
app = FastAPI(
    title       = "Cloudburst Prediction API",
    description = (
        "Real-time cloudburst probability forecasting for Uttarakhand, India, "
        "using a CNN-GAF framework trained on IMD AWS meteorological data. "
        "Based on the paper: Precision forecasting of cloudbursts with CNNs and GAF "
        "for real-time disaster response (Modeling Earth Systems and Environment, 2026). "
        "DOI: 10.1007/s40808-026-02826-4"
    ),
    version     = "1.0.0",
    docs_url    = "/docs",
    redoc_url   = "/redoc"
)


@app.get("/health", response_model=HealthResponse, tags=["System"])
def health_check():
    """Returns service status and model availability."""
    return HealthResponse(
        status        = "ok",
        models_loaded = True,
        timestamp     = datetime.datetime.utcnow().isoformat() + "Z"
    )


@app.post("/predict", response_model=PredictionResponse, tags=["Prediction"])
def predict(request: PredictionRequest):
    """
    Returns 6-hour and 12-hour cloudburst probabilities for a given IMD station.

    - **district**: IMD district name in uppercase (e.g. `DEHRADUN`)
    - **station**: IMD AWS station code (e.g. `MUSSOORIE(UKG)_UKG`)

    Risk levels:
    - `HIGH` — probability ≥ 75%
    - `MODERATE` — 40% ≤ probability < 75%
    - `LOW` — probability < 40%
    """
    return run_prediction(request.district, request.station)


@app.get("/stations", tags=["Reference"])
def list_stations():
    """
    Returns sample IMD station identifiers for Uttarakhand.
    Refer to the IMD AWS station registry for the full list.
    """
    return {
        "state": STATE,
        "sample_stations": [
            {"district": "DEHRADUN",   "station": "MUSSOORIE(UKG)_UKG"},
            {"district": "DEHRADUN",   "station": "DEHRADUN_UA"},
            {"district": "HARIDWAR",   "station": "HARIDWAR_UA"},
            {"district": "NAINITAL",   "station": "NAINITAL_UA"},
            {"district": "CHAMOLI",    "station": "JOSHIMATH_UA"},
            {"district": "RUDRAPRAYAG","station": "KEDARNATH_UA"},
        ],
        "note": "Station identifiers are IMD AWS codes. "
                "Contact IMD for the full Uttarakhand AWS station list."
    }


@app.exception_handler(Exception)
async def generic_exception_handler(request, exc):
    logger.error(f"Unhandled exception: {exc}")
    return JSONResponse(
        status_code=500,
        content={"detail": "An internal server error occurred."}
    )

## Start Server

The API will be available at:
- **Base URL:** http://localhost:8000
- **Interactive docs (Swagger):** http://localhost:8000/docs
- **Alternative docs (ReDoc):** http://localhost:8000/redoc

### Example request
```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"district": "DEHRADUN", "station": "MUSSOORIE(UKG)_UKG"}'
```

In [ ]:
def _run_server():
    uvicorn.run(app, host=API_HOST, port=API_PORT, log_level="info")

server_thread = Thread(target=_run_server, daemon=True)
server_thread.start()

print(f"\n✓ API running at http://localhost:{API_PORT}")
print(f"✓ Swagger docs: http://localhost:{API_PORT}/docs")